# 02 · BanglaBERT fine-tuning (3 seeds)
**Owner: Joy Das**

Fine-tunes `csebuetnlp/banglabert` on BanFakeNews-2.0 (headline + content as a sentence pair), picks the best epoch on the validation split, and evaluates on all test sets. Repeated for 3 seeds so the paper can report mean ± standard deviation.

Rough time on a Kaggle T4: about 20–30 min per epoch at `MAX_LEN = 256`, so the full run (3 seeds × 3 epochs) takes around 3–4.5 hours. Run it in the background with **Save & Run All**.

**Kaggle settings for this notebook** (right-hand panel → *Session options*):
- **Accelerator:** GPU T4 x2 (or P100)
- **Internet:** On (needed to download BanglaBERT and its text normalizer)
- **Inputs** (*Add Input*): `hrithikmajumdar/bangla-fake-news` (BanFakeNews-2.0) and your team's `banglafakenews2025` dataset. If you also have `New_Test_Dataset.csv` (the 1,000-article external test set of BanFakeNews-2.0), add it too; it is detected automatically.

Label convention everywhere: **1 = Real, 0 = Fake**. Run once with `FAST_DEV_RUN = True` (a few minutes) to check everything works, then set it to `False` and use **Save Version → Save & Run All** so it runs in the background.

In [1]:
!pip install -q git+https://github.com/csebuetnlp/normalizer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# ======================================================================
# COMMON DATA CODE  (identical in all 4 notebooks; if you change it,
# change it in every notebook, otherwise the example ids will not match)
# ======================================================================
import os, re, glob, json, random, hashlib, unicodedata, warnings
import numpy as np
import pandas as pd

SEED = 42
FAKE, REAL = 0, 1   # label convention everywhere (same as BanFakeNews-2.0 on Kaggle and BanglaFakeNews2025)

INPUT_DIR = os.environ.get('BFN_INPUT_DIR', '/kaggle/input')
OUT_DIR   = os.environ.get('BFN_OUT_DIR', '/kaggle/working')
PRED_DIR  = os.path.join(OUT_DIR, 'preds')
os.makedirs(PRED_DIR, exist_ok=True)

# Leave as None to auto-detect files under INPUT_DIR. Set a path (or a list of
# paths for bfn2025) only if auto-detection picks the wrong file.
PATHS = dict(
    bfn2_train=None, bfn2_val=None, bfn2_test=None,   # BanFakeNews-2.0 split files
    bfn2_all=None,                                    # ...or one single BanFakeNews-2.0 file
    bfn2_external=None,                               # New_Test_Dataset.csv (460 fake + 540 real)
    bfn2025=None,                                     # BanglaFakeNews2025 Bangla csv file(s)
)

LLM_SUBSET_SIZE  = 1000   # stratified sample of the BanFakeNews-2.0 test split used for ALL models
BFN2025_LLM_SIZE = 1000   # stratified sample of BanglaFakeNews2025 used for ALL models


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


_URL = re.compile(r'https?://\S+|www\.\S+')
_HTML = re.compile(r'<[^>]+>')
_WS = re.compile(r'\s+')

def clean_text(s):
    if not isinstance(s, str):
        s = '' if pd.isna(s) else str(s)
    s = unicodedata.normalize('NFC', s)
    s = _HTML.sub(' ', s)
    s = _URL.sub(' ', s)
    s = s.replace('\xa0', ' ')
    return _WS.sub(' ', s).strip()


_COLMAP = {'headline': 'Headline', 'title': 'Headline', 'heading': 'Headline',
           'content': 'Content', 'text': 'Content', 'article': 'Content', 'body': 'Content', 'news': 'Content',
           'label': 'Label', 'class': 'Label', 'target': 'Label',
           'category': 'Category', 'domain': 'Domain',
           'f-type': 'FType', 'f_type': 'FType', 'ftype': 'FType', 'fake_type': 'FType'}

def _std_cols(df, source):
    ren = {}
    for c in df.columns:
        k = str(c).strip().lower()
        if k in _COLMAP and _COLMAP[k] not in ren.values():
            ren[c] = _COLMAP[k]
    df = df.rename(columns=ren)
    missing = {'Content', 'Label'} - set(df.columns)
    if missing:
        raise ValueError(f'[{source}] columns {missing} not found. Columns are {list(df.columns)}. '
                         'Rename them or add them to _COLMAP.')
    if 'Headline' not in df.columns:
        df['Headline'] = ''
    return df


def _to_binary(df, source):
    lab = df['Label']
    if not pd.api.types.is_numeric_dtype(lab):
        low = lab.astype(str).str.strip().str.lower()
        m = {'real': REAL, 'authentic': REAL, 'true': REAL, '1': REAL,
             'fake': FAKE, 'false': FAKE, '0': FAKE}
        bad = set(low.unique()) - set(m)
        if bad:
            raise ValueError(f'[{source}] unknown label values {bad}')
        df['Label'] = low.map(m).astype(int)
        return df
    df = df[lab.notna()].copy()
    lab = df['Label'].astype(int)
    vals = set(lab.unique())
    if vals <= {0, 1}:
        df['Label'] = lab
        return df
    if vals == {0, 1, 2, 3}:
        # GitHub release of BanFakeNews-2.0: 3 = authentic, 0/1/2 = fake sub-types
        print(f'[{source}] 4-class labels found -> 3 = Real, 0/1/2 = Fake (sub-type kept in column FakeType)')
        df['FakeType'] = np.where(lab == 3, 'Not Applicable', 'type_' + lab.astype(str))
        df['Label'] = (lab == 3).astype(int)
        return df
    raise ValueError(f'[{source}] unexpected label values {vals}')


def _read(path):
    if path.lower().endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    return pd.read_csv(path, encoding='utf-8-sig')


def _bangla_ratio(texts):
    s = ''.join(texts)[:50000]
    letters = [ch for ch in s if ch.isalpha() or 'ঀ' <= ch <= '৿']
    return sum('ঀ' <= ch <= '৿' for ch in letters) / max(1, len(letters))


def _prepare(df, source):
    df = _std_cols(df.copy(), source)
    df = _to_binary(df, source)
    df['Headline'] = df['Headline'].map(clean_text)
    df['Content'] = df['Content'].map(clean_text)
    n0 = len(df)
    df = df[(df['Headline'].str.len() + df['Content'].str.len()) >= 20].copy()
    df['key'] = (df['Headline'] + '||' + df['Content']).map(lambda s: hashlib.md5(s.encode('utf-8')).hexdigest())
    df = df.drop_duplicates('key')
    if n0 - len(df):
        print(f'[{source}] dropped {n0 - len(df)} empty or duplicate rows')
    for col in ['Category', 'FType']:   # tidy labels like 'Crime ', 'Miscellaneous\t', 'Entertainmen'
        if col in df.columns:
            v = df[col].astype(str).str.strip()
            names = set(v.unique())
            fix = {}
            for a in names:
                longer = [b for b in names if b != a and b.startswith(a) and len(b) - len(a) <= 2]
                if len(longer) == 1:
                    fix[a] = longer[0]
            df[col] = v.replace(fix)
    keep = [c for c in ['Headline', 'Content', 'Label', 'Category', 'Domain', 'FType', 'FakeType', 'key']
            if c in df.columns]
    return df[keep].reset_index(drop=True)


def _all_files():
    fs = []
    for ext in ('csv', 'xlsx', 'xls'):
        fs += glob.glob(os.path.join(INPUT_DIR, '**', f'*.{ext}'), recursive=True)
    # never treat our own prediction files as data
    return sorted(f for f in fs if os.sep + 'preds' + os.sep not in f)


def _locate():
    found = {k: v for k, v in PATHS.items() if v}
    if isinstance(found.get('bfn2025'), str):
        found['bfn2025'] = [found['bfn2025']]
    files = _all_files()
    base = lambda f: os.path.basename(f).lower()
    if 'bfn2_external' not in found:
        c = [f for f in files if re.search(r'new[ _-]?test', base(f))]
        if c:
            found['bfn2_external'] = c[0]
    if 'bfn2025' not in found:
        c = [f for f in files if '2025' in f.lower()]
        if c:
            found['bfn2025'] = c
    used = {found.get('bfn2_external')} | set(found.get('bfn2025') or [])
    rest = [f for f in files if f not in used]
    for split, keys in [('bfn2_train', ['train']), ('bfn2_val', ['val', 'valid', 'dev']), ('bfn2_test', ['test'])]:
        if split not in found:
            c = [f for f in rest if any(k in base(f) for k in keys)]
            if c:
                found[split] = c[0]
    have_splits = all(k in found for k in ['bfn2_train', 'bfn2_val', 'bfn2_test'])
    if not have_splits and 'bfn2_all' not in found:
        c = [f for f in rest if f not in found.values()]
        if c:
            found['bfn2_all'] = max(c, key=os.path.getsize)   # biggest remaining file
    return found


def _strat_flag(df, n):
    from sklearn.model_selection import train_test_split
    if n is None or n >= len(df):
        return np.ones(len(df), dtype=bool)
    idx, _ = train_test_split(np.arange(len(df)), train_size=n, stratify=df['Label'], random_state=SEED)
    flag = np.zeros(len(df), dtype=bool)
    flag[idx] = True
    return flag


def load_all():
    """Returns dict: train, val, bfn2_test, and (if found) bfn2_external, bfn2025."""
    from sklearn.model_selection import train_test_split
    f = _locate()
    print('Files used:')
    for k, v in f.items():
        print(f'  {k}: {v}')
    if all(k in f for k in ['bfn2_train', 'bfn2_val', 'bfn2_test']):
        tr, va, te = (_prepare(_read(f[k]), k) for k in ['bfn2_train', 'bfn2_val', 'bfn2_test'])
    elif 'bfn2_all' in f:
        alldf = _prepare(_read(f['bfn2_all']), 'bfn2_all')
        tr, tmp = train_test_split(alldf, test_size=0.30, stratify=alldf['Label'], random_state=SEED)
        va, te = train_test_split(tmp, test_size=0.50, stratify=tmp['Label'], random_state=SEED)
        print('One BanFakeNews-2.0 file found -> stratified 70/15/15 split with seed 42 (same ratio as the paper)')
    else:
        raise FileNotFoundError(f'BanFakeNews-2.0 not found under {INPUT_DIR}. Add it with "Add Input".')

    # no article may appear in both training data and any evaluation set
    seen = set(tr['key'])
    va = va[~va['key'].isin(seen)]
    te = te[~te['key'].isin(seen | set(va['key']))]
    data = {'train': tr, 'val': va, 'bfn2_test': te}

    if 'bfn2_external' in f:
        data['bfn2_external'] = _prepare(_read(f['bfn2_external']), 'bfn2_external')
    if 'bfn2025' in f:
        parts = []
        for p in f['bfn2025']:
            d = _std_cols(_read(p), p)
            r = _bangla_ratio(d['Content'].astype(str).head(200).tolist())
            if r < 0.5:
                print(f'  skipping {os.path.basename(p)} (not Bangla text, Bangla ratio {r:.2f})')
                continue
            parts.append(d)
        if parts:
            data['bfn2025'] = _prepare(pd.concat(parts, ignore_index=True), 'bfn2025')

    for name in ['bfn2_external', 'bfn2025']:
        if name in data:
            n0 = len(data[name])
            data[name] = data[name][~data[name]['key'].isin(seen)]
            if n0 - len(data[name]):
                print(f'[{name}] removed {n0 - len(data[name])} articles that also appear in the training split')

    for name, df in data.items():
        df = df.reset_index(drop=True)
        df.insert(0, 'id', [f'{name}_{i:05d}' for i in range(len(df))])
        data[name] = df
    data['bfn2_test']['in_llm_subset'] = _strat_flag(data['bfn2_test'], LLM_SUBSET_SIZE)
    if 'bfn2_external' in data:
        data['bfn2_external']['in_llm_subset'] = True
    if 'bfn2025' in data:
        data['bfn2025']['in_llm_subset'] = _strat_flag(data['bfn2025'], BFN2025_LLM_SIZE)

    print('\nSplit sizes (fake / total):')
    for name, df in data.items():
        print(f'  {name:14s} {int((df.Label == FAKE).sum()):6d} / {len(df):6d}')
    for k in ['bfn2_external', 'bfn2025']:
        if k not in data:
            print(f'  NOTE: {k} not found -> it will be skipped. Add it as an input to include it.')
    return data


def eval_sets(data, llm_only=False):
    """Evaluation sets. The *_llm sets are the shared subsets every model is compared on."""
    te = data['bfn2_test']
    out = {} if llm_only else {'bfn2_test': te}
    out['bfn2_test_llm'] = te[te['in_llm_subset']]
    if 'bfn2_external' in data:
        out['bfn2_external'] = data['bfn2_external']
    if 'bfn2025' in data:
        d = data['bfn2025']
        if not llm_only:
            out['bfn2025'] = d
        out['bfn2025_llm'] = d[d['in_llm_subset']]
    return {k: v.reset_index(drop=True) for k, v in out.items()}


def compute_metrics(y_true, y_pred, p_fake=None):
    from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                                 matthews_corrcoef, roc_auc_score, average_precision_score)
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    m = dict(n=int(len(y_true)), n_fake=int((y_true == FAKE).sum()),
             macro_f1=f1_score(y_true, y_pred, average='macro', labels=[FAKE, REAL], zero_division=0),
             fake_f1=f1_score(y_true, y_pred, pos_label=FAKE, zero_division=0),
             fake_precision=precision_score(y_true, y_pred, pos_label=FAKE, zero_division=0),
             fake_recall=recall_score(y_true, y_pred, pos_label=FAKE, zero_division=0),
             real_f1=f1_score(y_true, y_pred, pos_label=REAL, zero_division=0),
             accuracy=accuracy_score(y_true, y_pred),
             mcc=matthews_corrcoef(y_true, y_pred),
             pred_fake_rate=float((y_pred == FAKE).mean()))
    if p_fake is not None and len(set(y_true)) == 2:
        is_fake = (y_true == FAKE).astype(int)
        m['roc_auc'] = roc_auc_score(is_fake, p_fake)
        m['pr_auc_fake'] = average_precision_score(is_fake, p_fake)
    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in m.items()}


def _slug(s):
    return re.sub(r'[^A-Za-z0-9.+\-]+', '-', s).strip('-')


def report(model, setting, seed, set_name, df, y_pred, p_fake=None, extra=None):
    """Prints metrics, appends them to results.jsonl and saves per-article predictions."""
    m = compute_metrics(df['Label'].values, y_pred, p_fake)
    rec = dict(model=model, setting=setting, seed=int(seed), eval_set=set_name, **m, **(extra or {}))
    with open(os.path.join(OUT_DIR, 'results.jsonl'), 'a', encoding='utf-8') as fh:
        fh.write(json.dumps(rec, ensure_ascii=False) + '\n')
    out = pd.DataFrame({'id': df['id'].values, 'y_true': df['Label'].values,
                        'y_pred': np.asarray(y_pred).astype(int)})
    if p_fake is not None:
        out['p_fake'] = np.round(np.asarray(p_fake, dtype=float), 5)
    out.to_csv(os.path.join(PRED_DIR, f'{_slug(model)}__{setting}__{set_name}__seed{seed}.csv'), index=False)
    auc = f"  AUC={m['roc_auc']:.4f}" if 'roc_auc' in m else ''
    print(f"{model:24s} {setting:12s} seed={seed:<3d} {set_name:14s} n={m['n']:5d}  "
          f"macroF1={m['macro_f1']:.4f}  fakeF1={m['fake_f1']:.4f}{auc}")
    return rec

In [3]:
import gc, time
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

MODEL_NAME   = 'csebuetnlp/banglabert'
MAX_LEN      = 256        # 512 is possible but about 2x slower
BATCH_SIZE   = 16
EVAL_BATCH   = 64
LR           = 2e-5
EPOCHS       = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEEDS        = [42, 43, 44]
USE_CLASS_WEIGHTS = True  # fake news is the minority class (about 20%)
NUM_WORKERS  = 2
FAST_DEV_RUN = False     # True = tiny run to check the code works

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

try:
    from normalizer import normalize as bn_normalize      # BanglaBERT was pre-trained on normalized text
except ImportError:
    bn_normalize = None
    print('WARNING: csebuetnlp normalizer is not installed, using plain text. Turn Internet on and re-run the pip cell.')

def norm(s, max_chars):
    s = s[:max_chars]
    return bn_normalize(s) if bn_normalize else s

device: cuda Tesla T4


## 1. Data and tokenization

In [4]:
data = load_all()
sets = eval_sets(data)
train_df, val_df = data['train'], data['val']
if FAST_DEV_RUN:
    train_df = pd.concat([train_df[train_df.Label == l].sample(200, random_state=SEED) for l in (REAL, FAKE)])
    val_df = val_df.sample(200, random_state=SEED)
    sets = {k: v.head(200) for k, v in sets.items()}
    EPOCHS, SEEDS = 1, [42]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_CHARS = MAX_LEN * 8   # only normalize/tokenize the part that can fit

def encode(df):
    t0 = time.time()
    heads = [norm(h, 300) for h in df['Headline']]
    bodies = [norm(c, MAX_CHARS) for c in df['Content']]
    enc = tokenizer(heads, bodies, truncation='longest_first', max_length=MAX_LEN)
    print(f'  encoded {len(df):,} articles in {time.time() - t0:.0f}s')
    return enc

print('train'); train_enc = encode(train_df)
print('val');   val_enc = encode(val_df)
set_encs = {}
for k, v in sets.items():
    print(k); set_encs[k] = encode(v)

class EncDS(Dataset):
    def __init__(self, enc, labels):
        self.enc, self.labels = enc, list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.enc.keys()}
        item['labels'] = int(self.labels[i])
        return item

def collate(batch):
    labels = torch.tensor([b.pop('labels') for b in batch])
    out = tokenizer.pad(batch, return_tensors='pt')
    out['labels'] = labels
    return out

train_ds = EncDS(train_enc, train_df['Label'])
counts = np.bincount(train_df['Label'], minlength=2)
class_w = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float)
print('class counts', counts, '-> class weights', class_w.numpy().round(3))

Files used:
  bfn2025: ['/kaggle/input/datasets/mahbubulalamabid/banglafakenews2025/Bangla_Fake_News_Dataset.xlsx', '/kaggle/input/datasets/mahbubulalamabid/banglafakenews2025/Bangla_Real_News_Dataset.xlsx']
  bfn2_train: /kaggle/input/datasets/hrithikmajumdar/bangla-fake-news/train_cleaned.csv
  bfn2_val: /kaggle/input/datasets/hrithikmajumdar/bangla-fake-news/val_cleaned.csv
  bfn2_test: /kaggle/input/datasets/hrithikmajumdar/bangla-fake-news/test_cleaned.csv
[bfn2_train] 4-class labels found -> 3 = Real, 0/1/2 = Fake (sub-type kept in column FakeType)
[bfn2_train] dropped 1510 empty or duplicate rows
[bfn2_val] 4-class labels found -> 3 = Real, 0/1/2 = Fake (sub-type kept in column FakeType)
[bfn2_val] dropped 75 empty or duplicate rows
[bfn2_test] 4-class labels found -> 3 = Real, 0/1/2 = Fake (sub-type kept in column FakeType)
[bfn2_test] dropped 83 empty or duplicate rows
[bfn2025] dropped 8 empty or duplicate rows

Split sizes (fake / total):
  train            7243 /  40870
  v

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

train
  encoded 40,870 articles in 150s
val
  encoded 8,370 articles in 31s
bfn2_test
  encoded 8,259 articles in 31s
bfn2_test_llm
  encoded 1,000 articles in 4s
bfn2025
  encoded 3,992 articles in 12s
bfn2025_llm
  encoded 1,000 articles in 3s
class counts [ 7243 33627] -> class weights [2.821 0.608]


## 2. Training and evaluation helpers

In [5]:
@torch.no_grad()
def predict_p_fake(model, enc):
    model.eval()
    n = len(enc['input_ids'])
    order = np.argsort([len(x) for x in enc['input_ids']])   # similar lengths together = faster
    p = np.zeros(n, dtype=np.float32)
    for s in range(0, n, EVAL_BATCH):
        idx = order[s:s + EVAL_BATCH]
        batch = tokenizer.pad([{k: enc[k][i] for k in enc.keys()} for i in idx], return_tensors='pt').to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits.float()
        p[idx] = torch.softmax(logits, -1)[:, FAKE].cpu().numpy()
    return p

def to_pred(p_fake):
    return np.where(p_fake >= 0.5, FAKE, REAL)

def train_one_seed(seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                        generator=g, num_workers=NUM_WORKERS)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total = len(loader) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total), total)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_w.to(DEVICE) if USE_CLASS_WEIGHTS else None)
    best_f1, best_state, best_epoch = -1.0, None, -1
    for epoch in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); running = 0.0
        for step, batch in enumerate(loader, 1):
            labels = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item()
            if step % 200 == 0 or step == len(loader):
                el = time.time() - t0
                print(f'  seed {seed} epoch {epoch} step {step}/{len(loader)} loss {running / step:.4f} '
                      f'({el / 60:.1f} min, ~{el / step * (len(loader) - step) / 60:.1f} min left in epoch)')
        pv = predict_p_fake(model, val_enc)
        f1 = f1_score(val_df['Label'].values, to_pred(pv), average='macro')
        print(f'seed {seed} epoch {epoch}: val macro-F1 = {f1:.4f}')
        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'seed {seed}: using epoch {best_epoch} (val macro-F1 {best_f1:.4f})')
    return model, best_epoch, best_f1

## 3. Run all seeds

In [6]:
for seed in SEEDS:
    t_start = time.time()
    model, best_epoch, val_f1 = train_one_seed(seed)
    for name, df in sets.items():
        p = predict_p_fake(model, set_encs[name])
        report('BanglaBERT', 'fine_tuned', seed, name, df, to_pred(p), p,
               extra={'best_epoch': best_epoch, 'val_macro_f1': round(val_f1, 4), 'max_len': MAX_LEN})
    print(f'seed {seed} done in {(time.time() - t_start) / 60:.1f} min\n')
    del model; gc.collect(); torch.cuda.empty_cache()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

  seed 43 epoch 1 step 200/2555 loss 0.6448 (0.7 min, ~8.4 min left in epoch)
  seed 43 epoch 1 step 400/2555 loss 0.5479 (1.4 min, ~7.7 min left in epoch)
  seed 43 epoch 1 step 600/2555 loss 0.4880 (2.1 min, ~7.0 min left in epoch)
  seed 43 epoch 1 step 800/2555 loss 0.4604 (2.9 min, ~6.3 min left in epoch)
  seed 43 epoch 1 step 1000/2555 loss 0.4432 (3.6 min, ~5.5 min left in epoch)
  seed 43 epoch 1 step 1200/2555 loss 0.4306 (4.3 min, ~4.8 min left in epoch)
  seed 43 epoch 1 step 1400/2555 loss 0.4204 (5.0 min, ~4.1 min left in epoch)
  seed 43 epoch 1 step 1600/2555 loss 0.4127 (5.7 min, ~3.4 min left in epoch)
  seed 43 epoch 1 step 1800/2555 loss 0.4077 (6.4 min, ~2.7 min left in epoch)
  seed 43 epoch 1 step 2000/2555 loss 0.4032 (7.1 min, ~2.0 min left in epoch)
  seed 43 epoch 1 step 2200/2555 loss 0.4000 (7.8 min, ~1.3 min left in epoch)
  seed 43 epoch 1 step 2400/2555 loss 0.3950 (8.5 min, ~0.6 min left in epoch)
  seed 43 epoch 1 step 2555/2555 loss 0.3910 (9.1 min, ~

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  seed 44 epoch 1 step 200/2555 loss 0.6436 (0.7 min, ~8.4 min left in epoch)
  seed 44 epoch 1 step 400/2555 loss 0.5506 (1.4 min, ~7.7 min left in epoch)
  seed 44 epoch 1 step 600/2555 loss 0.4926 (2.1 min, ~7.0 min left in epoch)
  seed 44 epoch 1 step 800/2555 loss 0.4661 (2.9 min, ~6.3 min left in epoch)
  seed 44 epoch 1 step 1000/2555 loss 0.4476 (3.6 min, ~5.5 min left in epoch)
  seed 44 epoch 1 step 1200/2555 loss 0.4316 (4.3 min, ~4.8 min left in epoch)
  seed 44 epoch 1 step 1400/2555 loss 0.4238 (5.0 min, ~4.1 min left in epoch)
  seed 44 epoch 1 step 1600/2555 loss 0.4157 (5.7 min, ~3.4 min left in epoch)
  seed 44 epoch 1 step 1800/2555 loss 0.4114 (6.4 min, ~2.7 min left in epoch)
  seed 44 epoch 1 step 2000/2555 loss 0.4077 (7.1 min, ~2.0 min left in epoch)
  seed 44 epoch 1 step 2200/2555 loss 0.4049 (7.8 min, ~1.3 min left in epoch)
  seed 44 epoch 1 step 2400/2555 loss 0.4010 (8.5 min, ~0.6 min left in epoch)
  seed 44 epoch 1 step 2555/2555 loss 0.3978 (9.1 min, ~

In [7]:
res = pd.read_json(os.path.join(OUT_DIR, 'results.jsonl'), lines=True)
res = res[res.model == 'BanglaBERT'].drop_duplicates(['seed', 'eval_set'], keep='last')
display(res.groupby('eval_set')[['macro_f1', 'fake_f1', 'roc_auc']].agg(['mean', 'std']).round(4))

macro_f1         fake_f1         roc_auc        
                  mean     std    mean     std    mean     std
eval_set                                                      
bfn2025         0.7942  0.0163  0.7773  0.0265  0.8640  0.0137
bfn2025_llm     0.8009  0.0146  0.7855  0.0235  0.8705  0.0086
bfn2_test       0.9237  0.0031  0.8686  0.0052  0.9640  0.0009
bfn2_test_llm   0.9100  0.0006  0.8447  0.0010  0.9541  0.0083